# Notebook 02: Exploratory Data Analysis (EDA) & Domain Insights

**Project**: Telecom Customer Churn Prediction & Retention System
**Task**: TASK 04 — Exploratory Data Analysis & Visual Analytics

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path
from IPython.display import Image, display

sys.path.append(str(Path.cwd().parent))

from src.eda import generate_eda_figures, compute_eda_summary_statistics
from src.utils import get_project_root


## 1. Load Cleaned Dataset & Execute EDA Pipeline
Ingesting `data/processed/cleaned_churn_data.csv`.

In [ ]:
project_root = get_project_root()
cleaned_csv = project_root / "data" / "processed" / "cleaned_churn_data.csv"
df = pd.read_csv(cleaned_csv)

figures = generate_eda_figures(processed_df_path=cleaned_csv)
print(f"Cleaned Dataset Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")


## 2. Target Class Distribution Analysis
Examining the overall churn rate across the customer base.

In [ ]:
display(Image(filename=str(figures['target_distribution'])))

churn_dist = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100
pd.DataFrame({'Count': churn_dist, 'Percentage (%)': churn_pct.round(2)})


### Business Observations — Target Class Distribution:
1. **Class Imbalance**: Out of 7,019 cleaned customer records, **5,163 customers (73.56%)** are retained (`No`), while **1,856 customers (26.44%)** churned (`Yes`).
2. **Metric Selection Implication**: Predicting all customers as `No` yields a naive 73.56% accuracy but catches 0% of churners. Consequently, **Recall**, **F1-Score**, and **ROC-AUC** must be prioritized over raw Accuracy.


## 3. Numerical Features & Churn Analysis
Evaluating distributions and box plots for `tenure`, `MonthlyCharges`, and `TotalCharges`.

In [ ]:
display(Image(filename=str(figures['numerical_distributions'])))

summary_stats = df.groupby('Churn')[['tenure', 'MonthlyCharges', 'TotalCharges']].agg(['mean', 'median', 'std']).round(2)
summary_stats


### Business Observations — Numerical Features vs Churn:
1. **Tenure Behavior**:
   - Churned customers have a significantly lower median tenure (**10.0 months**) compared to retained customers (**38.0 months**).
   - Peak churn occurs within the first **0 to 12 months** of customer subscription. High tenure customers ($\ge 48$ months) demonstrate high retention stability.
2. **Monthly Charges Impact**:
   - Churned customers exhibit higher average monthly bills (**Mean: $74.45 / Median: $79.65**) compared to retained customers (**Mean: $61.31 / Median: $64.45**).
   - Higher pricing tiers increase price sensitivity and customer defection rates.
3. **Total Charges Cumulative Effect**:
   - Retained customers accrue higher overall lifetime value (**Median: $1,682.00**) compared to churned customers (**Median: $703.55**), reflecting longer tenure duration.


## 4. Categorical Key Features vs Churn
Analyzing contractual, service, and demographic drivers of churn.

In [ ]:
display(Image(filename=str(figures['categorical_vs_churn'])))

contract_pct = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
internet_pct = pd.crosstab(df['InternetService'], df['Churn'], normalize='index') * 100

print("--- Churn Rate by Contract Type (%) ---")
print(contract_pct.round(2))

print("
--- Churn Rate by Internet Service Type (%) ---")
print(internet_pct.round(2))


### Business Observations — Categorical Key Drivers:
1. **Contract Type**:
   - **Month-to-Month Contracts**: High churn risk (**> 42% churn rate**). Represents the single strongest contractual indicator of customer churn.
   - **One-Year / Two-Year Contracts**: Extremely low churn rates (**< 11% for 1-year, < 3% for 2-year**). Long-term contracts effectively lock in customer retention.
2. **Internet Service Type**:
   - **Fiber Optic Customers**: Exhibit an elevated churn rate (**~41.8%**), indicating potential service quality complaints or competitive pricing pressure.
   - **DSL Customers**: Display significantly lower churn (**~18.9%**).
3. **Value-Added Service Add-ons**:
   - Customers subscribing to `TechSupport` and `OnlineSecurity` have dramatically lower churn rates (< 15%). These services act as strong ecosystem retention anchors.


## 5. Multivariate Analysis: Tenure vs Monthly Charges
Exploring decision boundaries between tenure and billing levels.

In [ ]:
display(Image(filename=str(figures['multivariate_tenure_charges'])))


### Business Observations — Multivariate Analysis:
1. **High Risk Cohort**: Short tenure ($< 15$ months) combined with high monthly charges ($> \$70/month$) forms the highest concentration of churned customers (red cluster).
2. **Loyal Cohort**: High tenure ($> 40$ months) customers remain retained regardless of monthly charge tier.
3. **Retention Strategy Takeaway**: Proactive retention campaigns should specifically target new customers ($< 12$ months) enrolled in Month-to-Month contracts with high monthly bills.


## 6. Correlation & Multicollinearity Analysis
Computing Pearson correlation matrix and Variance Inflation Factors (VIF).

In [ ]:
from src.correlation import run_correlation_analysis

corr_results = run_correlation_analysis()
display(Image(filename=str(root / "reports" / "figures" / "05_correlation_heatmap.png")))

print("--- Variance Inflation Factor (VIF) Scores ---")
for feat, vif in corr_results["vif_scores"].items():
    print(f" - {feat}: VIF = {vif}")

print("
--- Feature Correlations with Target (Churn) ---")
for feat, corr_val in corr_results["target_correlations_with_churn"].items():
    print(f" - {feat}: r = {corr_val}")


### Key Statistical Insights & Multicollinearity Recommendations:
1. **Strongest Single Predictor**: `tenure` has the strongest negative correlation with churn (**r = -0.3527**). Long-standing customers exhibit significantly higher retention loyalty.
2. **Billing Sensitivity**: `MonthlyCharges` displays positive correlation with churn (**r = +0.1937**). Higher monthly pricing increases churn risk.
3. **Multicollinearity Diagnostic**:
   - `TotalCharges` exhibits high VIF (**9.450**) and high correlation with `tenure` (**r = 0.8258**).
   - **Modeling Action Plan**:
     - Tree-based ensemble models (**Random Forest**, **XGBoost**, **LightGBM**) naturally handle multicollinearity and will retain all features.
     - Linear models (**Logistic Regression**) benefit from regularization penalty ($L1 / L2$) or ratio engineering (`AverageMonthlyChargeRatio = TotalCharges / (tenure + 1)`).
